In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv('./mushrooms.csv')

In [2]:
data.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [3]:
data.columns

Index(['class', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor',
       'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color',
       'stalk-shape', 'stalk-root', 'stalk-surface-above-ring',
       'stalk-surface-below-ring', 'stalk-color-above-ring',
       'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number',
       'ring-type', 'spore-print-color', 'population', 'habitat'],
      dtype='object')

In [4]:
len(data.nunique())

23

In [5]:
data.shape

(8124, 23)

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   class                     8124 non-null   object
 1   cap-shape                 8124 non-null   object
 2   cap-surface               8124 non-null   object
 3   cap-color                 8124 non-null   object
 4   bruises                   8124 non-null   object
 5   odor                      8124 non-null   object
 6   gill-attachment           8124 non-null   object
 7   gill-spacing              8124 non-null   object
 8   gill-size                 8124 non-null   object
 9   gill-color                8124 non-null   object
 10  stalk-shape               8124 non-null   object
 11  stalk-root                8124 non-null   object
 12  stalk-surface-above-ring  8124 non-null   object
 13  stalk-surface-below-ring  8124 non-null   object
 14  stalk-color-above-ring  

In [7]:
df = data.sample(frac=0.8,random_state=42)
df_test = data.drop(df.index)

In [8]:
def entropy(s):
    probs = s.value_counts(normalize=True)
    return -np.sum(probs * np.log2(probs))

In [9]:
def information_gain(df,split_attribute_name,target_attribute_name):
    # Calculate entropy
    total_entropy = entropy(df[target_attribute_name])
    
    # Get the unique value and counts for the split attribute considered
    vals,counts = np.unique(df[split_attribute_name],return_counts = True)
    
    weighted_entropy = 0 
    for i in range(len(vals)):
        sub_df = df[df[split_attribute_name] == vals[i]]
        sub_entropy = entropy(sub_df[target_attribute_name])
        weighted_entropy += (counts[i]/np.sum(counts)) * sub_entropy
    
    # Information Gain is the reduction in entropy
    return total_entropy - weighted_entropy

In [10]:
# 3. The Tree Node Structure

class Node:
    def __init__(self,feature=None,value=None,results=None,branches=None):
        self.feature = feature # The column name we split on
        self.value = value     # The specific value of the parent split
        self.results = results # If this is a leaf node, it holds the final prediction class
        self.branches = branches # Dictionary holding child nodes

In [11]:
def build_tree(df,target_attribute_name,features):
    target_values = df[target_attribute_name]
        
    # If all target values are identical , return a leaf node with that value
    if(len(np.unique(target_values)) <= 1):
        return Node(results = target_values.iloc[0])
        
    # If you run out of features to split on, return the majority vote
    if(len(features) == 0):
        majority_vote = target_values.value_counts().idxmax()
        return Node(results = majority_vote)
        
    # Step A : Find the best features to split on based on information gain
    gains = [information_gain(df,f,target_attribute_name) for f in features]
    best_feature_idx = np.argmax(gains)
    best_feature = features[best_feature_idx]
        
    # Create a parent node named after the best feature
    node = Node(feature=best_feature,branches={})
        
    # Remove the chosen feature from the pool of remaining features
    remaining_features = [f for f in features if f != best_feature]
        
    # Split the dataset into subsets for each unique value in the best feature
    feature_values = df[best_feature].unique()
        
    for val in feature_values:
        sub_df = df[df[best_feature]==val]
            
        # Recursively build the sub-tree down this branch
        child_node = build_tree(sub_df,target_attribute_name,remaining_features)
            
        node.branches[val] = child_node
    return node
def predict(node,sample):
    # If we hit a leaf node, return the result
    if node.results is not None:
        return node.results
        
    # Get the value of the sample for the feature this node splits on
    sample_val = sample[node.feature]
        
    # Traverse down the matching branch
    if sample_val in node.branches:
        return predict(node.branches[sample_val], sample)
    else:
        return None

In [12]:
target_attribute_name = 'class'

In [13]:
features = [col for col in df.columns if col != 'class']

In [14]:
tree = build_tree(df, target_attribute_name='class', features=features)

In [19]:
predicted = []

X_test = df_test.drop(columns=['class'])
y_test = df['class']
incorrect_classification = 0
for row, out in zip(X_test.to_dict('records'), y_test):
    pred = predict(tree, row)
    print(f"Predicted : {pred} ; Output : {out}")
    if(pred != out):
        incorrect_classification += 1
    predicted.append(pred)

Predicted : p ; Output : e
Predicted : e ; Output : p
Predicted : e ; Output : p
Predicted : e ; Output : e
Predicted : e ; Output : p
Predicted : e ; Output : p
Predicted : p ; Output : p
Predicted : e ; Output : p
Predicted : e ; Output : e
Predicted : e ; Output : e
Predicted : e ; Output : e
Predicted : e ; Output : p
Predicted : e ; Output : e
Predicted : p ; Output : e
Predicted : e ; Output : e
Predicted : e ; Output : e
Predicted : e ; Output : e
Predicted : e ; Output : p
Predicted : e ; Output : e
Predicted : e ; Output : e
Predicted : e ; Output : e
Predicted : e ; Output : e
Predicted : e ; Output : p
Predicted : e ; Output : e
Predicted : e ; Output : p
Predicted : e ; Output : e
Predicted : e ; Output : e
Predicted : e ; Output : e
Predicted : e ; Output : e
Predicted : e ; Output : p
Predicted : e ; Output : p
Predicted : e ; Output : p
Predicted : e ; Output : e
Predicted : e ; Output : e
Predicted : e ; Output : e
Predicted : e ; Output : p
Predicted : e ; Output : e
P

In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = data.drop(columns=['class'])
X = pd.get_dummies(X, drop_first=True)
y = data['class']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [30]:
rf_model = RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42)

# Train the model
rf_model.fit(X_train,y_train)

# Make Predictions
predictions = rf_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test,predictions)

print(f"Model Accuracy {accuracy}")

Model Accuracy 1.0


Index(['cap-shape_c', 'cap-shape_f', 'cap-shape_k', 'cap-shape_s',
       'cap-shape_x', 'cap-surface_g', 'cap-surface_s', 'cap-surface_y',
       'cap-color_c', 'cap-color_e', 'cap-color_g', 'cap-color_n',
       'cap-color_p', 'cap-color_r', 'cap-color_u', 'cap-color_w',
       'cap-color_y', 'bruises_t', 'odor_c', 'odor_f', 'odor_l', 'odor_m',
       'odor_n', 'odor_p', 'odor_s', 'odor_y', 'gill-attachment_f',
       'gill-spacing_w', 'gill-size_n', 'gill-color_e', 'gill-color_g',
       'gill-color_h', 'gill-color_k', 'gill-color_n', 'gill-color_o',
       'gill-color_p', 'gill-color_r', 'gill-color_u', 'gill-color_w',
       'gill-color_y', 'stalk-shape_t', 'stalk-root_b', 'stalk-root_c',
       'stalk-root_e', 'stalk-root_r', 'stalk-surface-above-ring_k',
       'stalk-surface-above-ring_s', 'stalk-surface-above-ring_y',
       'stalk-surface-below-ring_k', 'stalk-surface-below-ring_s',
       'stalk-surface-below-ring_y', 'stalk-color-above-ring_c',
       'stalk-color-above-rin